# Multi-Agent Systems

**What you will build:** a **supervisor** that routes each request to the right specialist agent, and each specialist handles its own domain. Maps to chapter 10 of the n8n course — including its counter-lesson about when *not* to use multiple agents.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/26_multi_agent_supervisor.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## Specialists as tools

The multi-agent pattern least likely to break across versions is also the simplest: build each specialist with `create_agent`, then expose each one to a **supervisor** as a tool. The supervisor is itself just an agent (2.0) whose tools happen to be other agents. Delegation is tool calling — the same protocol from 0.0, one level up.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# --- Specialist 1: math ---
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b
@tool
def percent_of(part: float, whole: float) -> float:
    """Return part as a percentage of whole."""
    return 100 * part / whole

math_agent = create_agent(model=llm, tools=[add, percent_of],
                          system_prompt="You are a math specialist. Use your tools to compute exact answers.")

# --- Specialist 2: research (fake knowledge) ---
@tool
def lookup_capital(country: str) -> str:
    """Return the capital city of a country."""
    caps = {"japan": "Tokyo", "france": "Paris", "spain": "Madrid"}
    return caps.get(country.lower(), "unknown")

research_agent = create_agent(model=llm, tools=[lookup_capital],
                              system_prompt="You are a research specialist. Look up facts with your tools.")

Now wrap each specialist as a delegation tool, and give both to the supervisor:

In [ ]:
@tool
def math_expert(query: str) -> str:
    """Delegate math and calculation questions to the math specialist."""
    out = math_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return out["messages"][-1].content

@tool
def research_expert(query: str) -> str:
    """Delegate factual lookup questions to the research specialist."""
    out = research_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return out["messages"][-1].content

supervisor = create_agent(
    model=llm, tools=[math_expert, research_expert],
    system_prompt="Route each part of the request to the right expert, then combine their answers.",
)

result = supervisor.invoke({"messages": [{"role": "user",
          "content": "What is 30 as a percentage of 240, and what's the capital of Japan?"}]})
print(result["messages"][-1].content)

The supervisor split the request, sent the math to `math_agent` and the fact to `research_agent`, and merged the results. Each specialist stays simple and focused; the supervisor only coordinates.

```{important}
**Multi-agent is not automatically better.** Every delegation is extra model calls, latency, and cost, and more places to go wrong. Reach for it only when domains are genuinely separate or one prompt has become an unmanageable list of rules. Very often a single well-built agent with good tools (Block 1) beats a committee. This is the same warning the n8n course gives in chapter 10.
```

::::{dropdown} 🔧 Under the hood: supervisor patterns
:color: info

There's also a dedicated `langgraph-supervisor` package (`create_supervisor(...)`) and a peer-to-peer `langgraph-swarm`. They add message-passing conventions and hand-off tooling on top of what you just built. The tools-as-delegation pattern here is the most transparent one, which is why we teach it first. For deep multi-agent systems, explore those packages once this pattern is second nature.
::::

```{note}
Two protocols, two levels. **MCP** (18) standardizes agent ↔ *tool*. **A2A** (Agent-to-Agent) standardizes agent ↔ *agent* — a wire format so agents from different teams or frameworks can delegate to each other over a network, instead of the in-process tool-delegation you built here. You rarely need it until agents live in separate services; then it's the MCP of multi-agent.
```

**Block 2 core complete.** You can now build stateful graphs: routing, the agent loop, persistence, human-in-the-loop, cyclic reflection, and multi-agent supervision — the tools for genuinely complex, controllable systems.

**What's next:** **27** is an optional capstone — RAG with a self-grading loop (agentic RAG) — and **28** adds **long-term memory** (remembering a user across separate conversations). Then Block 3 takes an agent to production.